In [ ]:
%%capture
!pip install --no-cache-dir -U langchain langchain-core langchain-community langchain-text-splitters langchain-chroma langchain-huggingface langchain-openrouter -q
!pip install --no-cache-dir -U sentence-transformers transformers huggingface-hub -q
!pip install --no-cache-dir -U chromadb -q
!pip install --no-cache-dir wget==3.2 -q
!pip install pypdf

In [2]:
# You can use this section to suppress warnings generated by your code:
# always restart session before running this
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.chains import RetrievalQA
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory
from langchain_community.document_loaders import PyPDFLoader

from langchain.chat_models import init_chat_model
import wget

In [3]:
filename = 'companyPolicies.txt'
url = ' https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/XVnuuEg94sAE4S_xAsGxBA.txt'

# Use wget to download the file
wget.download(url,out=filename)
print('file downloaded')

file downloaded


In [ ]:
with open(filename, 'r') as file:
    # Read the contents of the file
    contents = file.read()
    print(contents)

In [5]:
loader = TextLoader(filename)
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)
print(texts)

[Document(metadata={'source': 'companyPolicies.txt'}, page_content='Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  \n\nLast year COVID-19 kept us apart. This year we are finally together again. \n\nTonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. \n\nWith a duty to one another to the American people to the Constitution. \n\nAnd with an unwavering resolve that freedom will always triumph over tyranny. \n\nSix days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. \n\nHe thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. \n\nHe met the Ukrainian people. \n\nFrom President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determinati

In [6]:
embeddings = HuggingFaceEmbeddings()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
model = init_chat_model(model = 'openrouter:poolside/laguna-s-2.1:free')
docsearch = Chroma.from_documents(texts, embeddings) # creating a vector data base from the givig chunks of the texts

In [8]:
def retriever(document, query):
  document = PyPDFLoader(document).load()

  texts = text_splitter.split_documents(document)
  docsearch = Chroma.from_documents(texts, embeddings)

  qa = RetrievalQA.from_chain_type(llm = model,
                                 chain_type="refine",
                                 retriever=docsearch.as_retriever(),
                                 return_source_documents=False)
  result = qa.invoke(query)
  return result['result']

In [ ]:
import gradio as gr

rag = gr.Interface(fn = retriever, inputs=[gr.File(type='filepath', file_types = ['.pdf']),gr.Textbox(label = 'query',
                                                                          placeholder = 'Enter your query here ')],
                   outputs = gr.Textbox(label = 'Output',placeholder = 'your output here'))
rag.launch(debug = True)

In [ ]:
qa = RetrievalQA.from_chain_type(llm = model,
                                 chain_type = "stuff",
                                 retriever = docsearch.as_retriever(),
                                 return_source_documents = True)


query = "Can you summarize the document for me?"
qa.invoke(query)

In [18]:
qa = RetrievalQA.from_chain_type(llm = model,
                                 chain_type = "stuff",
                                 retriever = docsearch.as_retriever(),
                                 return_source_documents = False)

query = "Can I eat in company vehicles?"
qa.invoke(query)

{'query': 'Can I eat in company vehicles?',
 'result': "I don't know. The provided text does not contain any information about company vehicle policies or rules regarding eating in them."}

In [19]:
prompt_template = """Use the information from the document to answer the question at the end. If you don't know the answer, just say that you don't know, definately do not try to make up an answer.

{context}

Question: {question}
"""

PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)

chain_type_kwargs = {"prompt": PROMPT}

In [21]:
query = "What I cannot do in it?"
qa.invoke(query)

{'query': 'What I cannot do in it?',
 'result': 'I don\'t know. The question "What I cannot do in it?" is too vague—the document contains multiple unrelated sections (a speech about Intel\'s planned Ohio semiconductor site, fee deposit slips for GIK Institute, and a political speech excerpt), and it\'s unclear what "it" refers to or what specific limitations you\'re asking about. The text doesn\'t explicitly state restrictions or things one cannot do in any clear context.'}

In [22]:
memory = ConversationBufferMemory(memory_key = "chat_history", return_message = True)

In [50]:
Qa = ConversationalRetrievalChain.from_llm(llm=model,
                                           chain_type="map_reduce",
                                           retriever=docsearch.as_retriever(),
                                           memory = memory,
                                           get_chat_history=lambda h : h,
                                           return_source_documents=False)
type(qa)

function

In [30]:
history = []
query = "What is mobile policy?"

result = qa.invoke({"question":query ,"chat_history": history})

print(result["answer"])

I don't know. The provided context does not contain any information about "mobile policy." It consists of excerpts from a statistical learning textbook covering topics such as local regression, smoothing splines, classification, confusion matrices, ROC curves, and principal component analysis.


In [ ]:
history.append((query, result["answer"]))
query = "List points in it?"

result = Qa.invoke({"question": query,"chat_history": history})
print(result["answer"])

history.append((query, result["answer"]))
query = "What is the aim of it?"

# result = Qa.invoke({"question": query,"chat_history": history})
# print(result["answer"])

In [58]:
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage

chat_history = InMemoryChatMessageHistory(
    messages=[
        message
        for user, ai in history
        for message in (
            HumanMessage(content=user),
            AIMessage(content=ai)
        )
    ]
)

In [76]:
def qa(question):
    memory = ConversationBufferMemory(memory_key = "chat_history", return_message = True)

    qa = ConversationalRetrievalChain.from_llm(llm = model,
                                               chain_type ="stuff",
                                               retriever = docsearch.as_retriever(),
                                               memory = memory,
                                               get_chat_history = lambda h:h,
                                               return_source_documents = False)
    history = []
    while True:
        query = question

        if query.lower() in ["quit","exit","bye"]:
            print("Answer: Goodbye!")
            break

        result = qa({"question": query ,"chat_history": history})

        history.append((query, result["answer"]))

        print(f"Answer: , {result["answer"]}")

In [79]:
interface = gr.Interface(fn = qa, inputs = 'text', outputs = 'text')
interface.launch(debug = True)

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://bb170579ff75b4b8c5.gradio.live
